# Microsoft Agent Framework — Lab 3: Workflows

This lab introduces **Workflows** — the graph-based orchestration layer in Microsoft Agent Framework.

While an `Agent` makes autonomous decisions via an LLM loop, a **Workflow** gives you **explicit control** over execution order, routing, and parallelism.

| Use an Agent when… | Use a Workflow when… |
|---|---|
| The task is open-ended / conversational | The process has well-defined steps |
| You need autonomous tool use | You need explicit control over execution order |
| A single LLM call (possibly with tools) suffices | Multiple agents or functions must coordinate |

### Key concepts
- **Executor** — a processing unit (custom logic or an AI agent)
- **Edge** — a connection between executors
- **WorkflowBuilder** — assembles executors + edges into a directed graph
- **WorkflowContext** — the runtime context for sending messages and yielding outputs

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

---
## Part 1 — Your First Executor

Executors inherit from `Executor` and use the `@handler` decorator on methods.  
The method's **type annotation** tells the runtime which message types it accepts.

In [1]:
from agent_framework import Executor, WorkflowContext, WorkflowBuilder, handler


class UpperCaseExecutor(Executor):
    """Transforms any text message to UPPER CASE."""

    def __init__(self):
        super().__init__(id="upper_case")

    @handler
    async def process(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text.upper())


class ExclamationExecutor(Executor):
    """Appends !!!! to whatever it receives."""

    def __init__(self):
        super().__init__(id="exclamation")

    @handler
    async def process(self, text: str, ctx: WorkflowContext[None, str]) -> None:
        await ctx.yield_output(text + "!!!!")

In [2]:
upper = UpperCaseExecutor()
exclaim = ExclamationExecutor()

# Build the workflow:  input → upper → exclaim → output
builder = WorkflowBuilder(start_executor=upper)
builder.add_edge(upper, exclaim)
workflow = builder.build()

events = await workflow.run("hello from microsoft agent framework")
print(events.get_outputs())

['HELLO FROM MICROSOFT AGENT FRAMEWORK!!!!']


---
## Part 2 — Function-Based Executors

Use the `@executor` decorator to create an executor from a plain async function — no class needed.

In [4]:
from agent_framework import executor as executor_decorator


@executor_decorator(id="word_counter")
async def word_counter(text: str, ctx: WorkflowContext[None, str]) -> None:
    """Counts words and yields a summary."""
    count = len(text.split())
    await ctx.yield_output(f"Input had {count} words.")


builder2 = WorkflowBuilder(start_executor=word_counter)
workflow2 = builder2.build()

result = await workflow2.run("The quick brown fox jumps over the lazy dog")
print(result.get_outputs())

['Input had 9 words.']


---
## Part 3 — AI Agents as Executors

The real power comes from wrapping AI agents inside executors.  
The executor calls `agent.run()` and forwards the result to the next step.

In [3]:
from agent_framework.openai import OpenAIChatCompletionClient

client = OpenAIChatCompletionClient(model="gpt-4o-mini")

In [4]:
class SummarizerExecutor(Executor):
    """Summarises an article to 3 bullet points."""

    def __init__(self):
        super().__init__(id="summarizer")
        self._agent = client.as_agent(
            name="summarizer",
            instructions="You summarise text into exactly 3 concise bullet points.",
        )

    @handler
    async def summarize(self, text: str, ctx: WorkflowContext[str]) -> None:
        result = await self._agent.run(f"Summarise this:\n\n{text}")
        await ctx.send_message(str(result))


class TranslatorExecutor(Executor):
    """Translates text to French."""

    def __init__(self):
        super().__init__(id="translator")
        self._agent = client.as_agent(
            name="translator",
            instructions="You translate text to French. Keep the same formatting.",
        )

    @handler
    async def translate(self, text: str, ctx: WorkflowContext[None, str]) -> None:
        result = await self._agent.run(f"Translate to French:\n\n{text}")
        await ctx.yield_output(str(result))

In [7]:
from IPython.display import display, Markdown

summarizer = SummarizerExecutor()
translator = TranslatorExecutor()

# Pipeline: input → summarizer → translator → output
builder3 = WorkflowBuilder(start_executor=summarizer)
builder3.add_edge(summarizer, translator)
workflow3 = builder3.build()

article = """
Microsoft Agent Framework is the next generation of both Semantic Kernel and AutoGen.
It combines AutoGen's simple agent abstractions with Semantic Kernel's enterprise features,
including session-based state management, type safety, middleware, and telemetry.
The framework introduces graph-based workflows that give developers explicit control
over multi-agent execution paths. It supports Microsoft Foundry, Azure OpenAI, OpenAI,
Anthropic, GitHub Copilot, Ollama, and Amazon Bedrock as inference providers.
Checkpointing allows long-running workflows to be resumed from a saved state.
"""

result = await workflow3.run(article)
display(Markdown(result.get_outputs()[0]))

- Le cadre Microsoft Agent combine les abstractions d'agents d'AutoGen avec les capacités d'entreprise de Semantic Kernel telles que la gestion des sessions, la sécurité des types et les middleware.
- Il propose des flux de travail basés sur des graphes pour contrôler les chemins d'exécution multi-agents et prend en charge les principaux fournisseurs d'inférence AI.
- Le cadre inclut le point de contrôle pour reprendre des flux de travail longs à partir d'états sauvegardés.

---
## Part 4 — Streaming Workflow Output

Pass `stream=True` to get workflow events as they arrive instead of waiting for everything.

In [8]:
print("Streaming workflow events...\n")

async for event in workflow3.run(article, stream=True):
    if event.type == "executor_completed":
        print(f"  ✓ Executor '{event.executor_id}' completed")
    elif event.type == "output":
        print(f"\n--- Final Output ---")
        display(Markdown(event.data))

Streaming workflow events...

  ✓ Executor 'summarizer' completed

--- Final Output ---


- Le Microsoft Agent Framework fusionne des fonctionnalités du Semantic Kernel et d'AutoGen, améliorant les capacités des agents avec des fonctionnalités d'entreprise.
- Il utilise des flux de travail basés sur des graphes pour un meilleur contrôle sur l'exécution multi-agent et prend en charge divers fournisseurs d'inférence comme Microsoft Foundry et OpenAI.
- La fonctionnalité de point de contrôle permet la reprise de flux de travail longs, assurant la continuité à partir d'un état enregistré.

  ✓ Executor 'translator' completed


---
## Part 5 — Rock, Paper, Scissors with Workflows

A fun example: 3 executor nodes — Player1, Player2, and a Judge that coordinates them.

The Judge sends messages **to** Player1 and Player2, collects their moves, then decides the winner.

In [9]:
class Player1Executor(Executor):
    """Chooses a move in Rock, Paper, Scissors."""

    def __init__(self):
        super().__init__(id="player1")
        self._agent = client.as_agent(
            name="player1",
            instructions="You are Player 1. Respond only with a single word: rock, paper, or scissors.",
        )

    @handler
    async def play(self, prompt: str, ctx: WorkflowContext[str]) -> None:
        result = await self._agent.run(prompt)
        await ctx.send_message(f"Player 1: {result}")


class Player2Executor(Executor):
    """Chooses a move in Rock, Paper, Scissors."""

    def __init__(self):
        super().__init__(id="player2")
        self._agent = client.as_agent(
            name="player2",
            instructions="You are Player 2. Respond only with a single word: rock, paper, or scissors.",
        )

    @handler
    async def play(self, prompt: str, ctx: WorkflowContext[str]) -> None:
        result = await self._agent.run(prompt)
        await ctx.send_message(f"Player 2: {result}")


class JudgeExecutor(Executor):
    """Sends prompts to players and judges the result."""

    def __init__(self, player1: Player1Executor, player2: Player2Executor):
        super().__init__(id="judge")
        self._player1 = player1
        self._player2 = player2
        self._judge_agent = client.as_agent(
            name="judge",
            instructions="You are judging rock, paper, scissors. State the winner clearly.",
        )

    @handler
    async def judge(self, start_signal: str, ctx: WorkflowContext[None, str]) -> None:
        instruction = "Choose: rock, paper, or scissors. One word only."

        # Directly call player executors (within the same workflow context)
        p1_result = await self._player1._agent.run(instruction)
        p2_result = await self._player2._agent.run(instruction)

        game_state = f"Player 1 chose: {p1_result}\nPlayer 2 chose: {p2_result}"
        verdict = await self._judge_agent.run(
            f"Rock, paper, scissors result:\n{game_state}\n\nWho wins?"
        )
        await ctx.yield_output(f"{game_state}\n\nVerdict: {verdict}")

In [10]:
p1 = Player1Executor()
p2 = Player2Executor()
judge = JudgeExecutor(p1, p2)

rps_builder = WorkflowBuilder(start_executor=judge)
rps_workflow = rps_builder.build()

result = await rps_workflow.run("Go!")
print(result.get_outputs()[0])

Player 1 chose: rock
Player 2 chose: scissors

Verdict: Player 1 wins! Rock crushes scissors.


---
### Summary

| Concept | Code |
|---|---|
| Custom executor | `class MyExec(Executor): @handler async def handle(self, msg: str, ctx: WorkflowContext) -> None` |
| Function executor | `@executor(id="...") async def my_fn(msg: str, ctx: WorkflowContext)` |
| Build workflow | `WorkflowBuilder(start_executor=...).add_edge(...).build()` |
| Run workflow | `await workflow.run(input_message)` → `result.get_outputs()` |
| Stream workflow | `async for event in workflow.run(input, stream=True)` |
| Yield final output | `await ctx.yield_output(...)` |
| Pass to next executor | `await ctx.send_message(...)` |

In **Lab 4** we build a complex fan-out / fan-in multi-agent workflow.